# Continuous-Prefix Table-CNN + Qwen3

This notebook trains the single-decoder architecture with no cross-attention:

`table cells → Qwen embeddings → mean pooling → MLP → 2D CNN → Qwen prefix embeddings → Qwen decoder`

Select a GPU runtime before starting. Training artifacts are mirrored to Google Drive, and rerunning the training cell resumes from the latest complete checkpoint.

In [ ]:
# Check that Colab assigned a GPU.
!nvidia-smi

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/seungjun-green/cnn_qwen_table_mcr.git"
REPO_DIR = Path("/content/table-cnn-mrc")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
print(f"Repository ready at {REPO_DIR}")

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded from Colab secrets.")
else:
    print("HF_TOKEN was not set. The public model and dataset may still download normally.")

DRIVE_ROOT = Path("/content/drive/MyDrive/cnn_qwen_table_mcr/outputs")
DRIVE_OUTPUT = DRIVE_ROOT / "continuous_prefix"
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Persistent training output: {DRIVE_OUTPUT}")

## Smoke test

This loads one real example and verifies loss, gradient flow through the cell encoder/CNN/projector, and generation before a long training run.

In [ ]:
subprocess.run(
    [
        "python",
        str(REPO_DIR / "scripts/smoke_test.py"),
        "--config",
        str(REPO_DIR / "configs/continuous_prefix.yaml"),
    ],
    cwd=REPO_DIR,
    check=True,
)

## Train or resume

The config allows up to 10 epochs with early-stopping patience 3. A full checkpoint is saved every 100 optimizer steps and after every epoch. If Colab disconnects, rerun the setup, Drive, and training cells; the latest Drive checkpoint is restored automatically. Do not delete `checkpoint_last.pt` when resuming.

In [ ]:
subprocess.run(
    [
        "python",
        str(REPO_DIR / "scripts/run_experiment.py"),
        "--config",
        str(REPO_DIR / "configs/continuous_prefix.yaml"),
        "--mirror-output-dir",
        str(DRIVE_OUTPUT),
    ],
    cwd=REPO_DIR,
    check=True,
)

## Inspect training history

In [ ]:
import json
import pandas as pd

history_path = DRIVE_OUTPUT / "history.json"
if not history_path.is_file():
    raise FileNotFoundError(f"Training history not found: {history_path}")

with history_path.open() as handle:
    history = json.load(handle)

print("Status:", history.get("status"))
print("Best validation Exact Match:", history.get("best_exact_match"))
epoch_rows = []
for epoch in history.get("epochs", []):
    validation = epoch.get("validation", {})
    epoch_rows.append({
        "epoch": epoch.get("epoch"),
        "training_loss": epoch.get("training_loss"),
        "validation_EM": validation.get("exact_match"),
        "best_EM": epoch.get("best_exact_match"),
    })
display(pd.DataFrame(epoch_rows))

## Verify that the trained model uses its table

This compares 200 validation examples with their correct tables and shuffled tables using the best checkpoint. A useful table representation should produce a positive Exact Match drop after shuffling.

In [ ]:
DIAGNOSTIC_OUTPUT = DRIVE_ROOT / "diagnostics/continuous_prefix"
subprocess.run(
    [
        "python",
        str(REPO_DIR / "scripts/diagnose_saved_runs.py"),
        "--configs",
        str(REPO_DIR / "configs/continuous_prefix.yaml"),
        "--mirror-root",
        str(DRIVE_ROOT),
        "--max-examples",
        "200",
        "--modes",
        "trained",
        "--checkpoint",
        "best",
        "--output-dir",
        str(DIAGNOSTIC_OUTPUT),
    ],
    cwd=REPO_DIR,
    check=True,
)

In [ ]:
with (DIAGNOSTIC_OUTPUT / "summary.json").open() as handle:
    diagnostic = json.load(handle)

metrics = diagnostic["runs"]["continuous_prefix"]["modes"]["trained"]
display(pd.DataFrame([{
    "correct_table_EM": metrics["correct_table_exact_match"],
    "shuffled_table_EM": metrics["shuffled_table_exact_match"],
    "EM_drop": metrics["exact_match_drop"],
    "prediction_change_rate": metrics["prediction_change_rate"],
    "correct_to_wrong": metrics["correct_to_wrong_count"],
    "examples": metrics["number_evaluated"],
}]).style.format({
    "correct_table_EM": "{:.2%}",
    "shuffled_table_EM": "{:.2%}",
    "EM_drop": "{:.2%}",
    "prediction_change_rate": "{:.2%}",
}))